In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from datetime import date
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
from IPython.display import display


## Setting file paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'Census'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())


def re_remove_post(x, exp = '.'):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0]


def clean_fips(df):
        
        if 'State FIPS' in df.columns:
            df['State FIPS'] = df['State FIPS'].astype(str).apply('{:0>2}'.format)
        if 'Place ID' in df.columns:
            df['Place ID'] = df['Place ID'].astype(str).apply('{:0>5}'.format)
        if 'Block Group ID' in df.columns:
            df['Block Group ID'] = df['Block Group ID'].astype(str).apply(re_remove_post).apply('{:0>1}'.format)
        if 'Tract ID' in df.columns:
            df['Tract ID'] = df['Tract ID'].astype(str).apply('{:0>6}'.format)
        if 'County FIPS' in df.columns:
            df['County FIPS'] = df['County FIPS'].astype(str).apply('{:0>3}'.format)
        if 'Congressional District' in df.columns:
            df['Congressional District'] = df['Congressional District'].astype(str).apply('{:0>2}'.format)
        if 'State Legislative Upper District' in df.columns:
            df['State Legislative Upper District'] = df['State Legislative Upper District'].astype(str).apply('{:0>3}'.format)
        if 'State Legislative Lower District' in df.columns:
            df['State Legislative Lower District'] = df['State Legislative Lower District'].astype(str).apply('{:0>3}'.format)

        return df


        

In [ ]:


path_in = path_main / 'Vibrant and Inclusive Places' / 'Economy' / 'Income' / 'Income_1 HH Income'
file_name = 'Income_1 Block Groups ACS5.xlsx'
file_in = path_in / file_name
sheet_name = 'Block Groups'

df_income = pd.read_excel(file_in, sheet_name=sheet_name)
df_income.head()



In [ ]:

df_income1 = df_income.copy()

df_income1 = df_income1[~df_income1['Block Group ID'].isna()]
df_income1 = df_income1.fillna(0)

df_income1 = clean_fips(df_income1)
df_income1['GEOID'] = df_income1['State FIPS'] + df_income1['County FIPS'] + df_income1['Tract ID'] + df_income1['Block Group ID']
df_income1['GEOID'] = df_income1['GEOID'].astype('int64')

df_income1 = df_income1[['GEOID', 'NAME', 'Median Household Income']]

df_income1


In [ ]:
path_out = path_prod / 'StoryMaps' / 'Housing'
workbook = 'median_household_income_blockgroups_ACS5.csv'
export=True

df = df_income1.copy()

# Export function
def export_to_csv():
    global workbook
    df.columns = [x.lower() for x in df.columns]
    df.columns = [re.sub('[^\\w\\s]', '_', col.strip()) for col in df.columns]
    df.columns = [re.sub('[\s+]'    , '_', col.strip()) for col in df.columns]
    df.columns = [re.sub('\\?'      , '' , col.strip()) for col in df.columns]
    workbook = re.sub('[\s+]', '_'   , workbook)
    workbook = re.sub('.xlsx', '.csv', workbook)
    workbook = workbook.lower()

    file_out = path_out / workbook
    print('Exporting here: ', path_out)
    print('Name of export: ', workbook)
    display(df.head())
    if export:
        df.to_csv(file_out, index=False)


export_to_csv()

